In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/dirty_financial_transactions.csv')

# Display the first 5 rows to understand the data structure
display(df.head())

,Transaction_ID,Transaction_Date,Customer_ID,Product_Name,Quantity,Price,Payment_Method,Transaction_Status
0,T0001,2024-08-02,C2205,Headphones,-5.0,$420.21,pay pal,NaN
1,T0002,2020-02-10,C3156,Coffee,469.0,-445.34202525395585,creditcard,Pending
2,T0003,2025-02-30,C2919,Tablet,-4.0,810.9930123946459,credit card,completed
3,T0004,2020-08-17,C3009,Tab,-7.0,868.6083413217348,PayPal,Pending
4,T0005,2025-02-30,C3488,Coffee Machine,-10.0,-763.1224490039416,PayPal,completed


In [2]:
# Check data types and non-null values
df.info()

# Check for missing values
print('\nMissing values per column:')
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 8 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Transaction_ID      94982 non-null   object 
 1   Transaction_Date    95120 non-null   object 
 2   Customer_ID         95122 non-null   object 
 3   Product_Name        100000 non-null  object 
 4   Quantity            94981 non-null   float64
 5   Price               66503 non-null   object 
 6   Payment_Method      100000 non-null  object 
 7   Transaction_Status  83321 non-null   object 
dtypes: float64(1), object(7)
memory usage: 6.1+ MB

Missing values per column:
Transaction_ID         5018
Transaction_Date       4880
Customer_ID            4878
Product_Name              0
Quantity               5019
Price                 33497
Payment_Method            0
Transaction_Status    16679
dtype: int64


### Handle Duplicates

We will check for and remove any duplicate rows in the dataset. Duplicates can skew analysis and should be addressed early in the cleaning process.

In [3]:
# Check for duplicate rows
duplicate_rows = df.duplicated().sum()
print(f"Number of duplicate rows found: {duplicate_rows}")

# Remove duplicate rows if any
if duplicate_rows > 0:
    df.drop_duplicates(inplace=True)
    print(f"Duplicate rows removed. New DataFrame shape: {df.shape}")
else:
    print("No duplicate rows found.")

Number of duplicate rows found: 994
Duplicate rows removed. New DataFrame shape: (99006, 8)


### Clean Text Data

We will clean categorical text columns to ensure consistency. This often involves converting text to a uniform case (e.g., lowercase) and stripping leading/trailing whitespace. We'll focus on `Payment_Method` and `Transaction_Status` for now, as they are likely candidates for such issues.

In [4]:
# Clean 'Payment_Method' column
df['Payment_Method'] = df['Payment_Method'].str.lower().str.strip()

# Clean 'Transaction_Status' column
df['Transaction_Status'] = df['Transaction_Status'].str.lower().str.strip()

# Display unique values to verify cleaning for Payment_Method
print('Unique values in Payment_Method after cleaning:')
print(df['Payment_Method'].unique())

# Display unique values to verify cleaning for Transaction_Status
print('\nUnique values in Transaction_Status after cleaning:')
print(df['Transaction_Status'].unique())

Unique values in Payment_Method after cleaning:
['pay pal' 'creditcard' 'credit card' 'paypal' 'cash']

Unique values in Transaction_Status after cleaning:
[nan 'pending' 'completed' 'complete' 'failed']


### Standardize Categorical Values

Following the initial text cleaning, we've identified some categories that are semantically the same but have different textual representations. We will standardize these by mapping them to a single, consistent value. This will ensure that these categories are correctly grouped when analyzed in Tableau.

In [5]:
# Standardize 'Payment_Method' values
df['Payment_Method'] = df['Payment_Method'].replace({
    'pay pal': 'paypal',
    'credit card': 'creditcard'
})

# Standardize 'Transaction_Status' values
df['Transaction_Status'] = df['Transaction_Status'].replace({
    'complete': 'completed'
})

# Display unique values to verify standardization for Payment_Method
print('Unique values in Payment_Method after standardization:')
print(df['Payment_Method'].unique())

# Display unique values to verify standardization for Transaction_Status
print('\nUnique values in Transaction_Status after standardization:')
print(df['Transaction_Status'].unique())

Unique values in Payment_Method after standardization:
['paypal' 'creditcard' 'cash']

Unique values in Transaction_Status after standardization:
[nan 'pending' 'completed' 'failed']


### Date/Time Conversion

We need to convert the `Transaction_Date` column to a proper datetime format. This is crucial for any time-series analysis and for Tableau to recognize it as a date field. We will also handle any dates that cannot be parsed into a valid datetime format.

In [6]:
# Convert 'Transaction_Date' to datetime, coercing errors
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'], errors='coerce')

# Check for unparseable dates that turned into NaT
print(f"Number of unparseable dates (converted to NaT): {df['Transaction_Date'].isnull().sum()}")

# For simplicity in this example, we'll drop rows where Transaction_Date is NaT.
# In a real scenario, you might impute or investigate further.
initial_rows = df.shape[0]
df.dropna(subset=['Transaction_Date'], inplace=True)
print(f"Dropped {initial_rows - df.shape[0]} rows with invalid Transaction_Date. New DataFrame shape: {df.shape}")

# Display the data types again to confirm conversion
df.info()

Number of unparseable dates (converted to NaT): 67566
Dropped 67566 rows with invalid Transaction_Date. New DataFrame shape: (31440, 8)
<class 'pandas.core.frame.DataFrame'>
Index: 31440 entries, 0 to 99998
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Transaction_ID      29860 non-null  object        
 1   Transaction_Date    31440 non-null  datetime64[ns]
 2   Customer_ID         29922 non-null  object        
 3   Product_Name        31440 non-null  object        
 4   Quantity            29891 non-null  float64       
 5   Price               20849 non-null  object        
 6   Payment_Method      31440 non-null  object        
 7   Transaction_Status  26225 non-null  object        
dtypes: datetime64[ns](1), float64(1), object(6)
memory usage: 2.2+ MB


### Clean and Validate Numerical Data

We need to convert the `Price` column to a numeric data type, as it's currently an object and contains non-numeric characters. We'll also handle its missing values. For the `Quantity` column, we'll validate for potentially incorrect negative values, which could indicate data entry errors or returns that need specific handling.

In [8]:
# Clean 'Price' column: remove non-numeric characters and convert to float
df['Price'] = df['Price'].astype(str).str.replace(r'[$,]', '', regex=True)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# Check for unparseable prices that turned into NaNs
print(f"Number of unparseable prices (converted to NaN): {df['Price'].isnull().sum()}")

# Impute missing 'Price' values. For simplicity, we'll use the median.
# A more sophisticated approach might involve predictive modeling.
median_price = df['Price'].median()
df['Price'] = df['Price'].fillna(median_price)
print(f"Missing 'Price' values imputed with median: {median_price:.2f}")

# Validate 'Quantity': check for negative values
negative_quantities = df[df['Quantity'] < 0].shape[0]
print(f"Number of transactions with negative quantity: {negative_quantities}")

# For this analysis, we'll assume negative quantities are invalid and convert them to their absolute value.
# In a real-world scenario, you might want to treat these as returns.
df['Quantity'] = df['Quantity'].abs()
print("Negative quantities converted to absolute values.")

# Impute missing 'Quantity' values with the median.
median_quantity = df['Quantity'].median()
df['Quantity'] = df['Quantity'].fillna(median_quantity)
print(f"Missing 'Quantity' values imputed with median: {median_quantity:.2f}")


# Display data types and info again to confirm changes
df.info()

Number of unparseable prices (converted to NaN): 0
Missing 'Price' values imputed with median: -54.40
Number of transactions with negative quantity: 0
Negative quantities converted to absolute values.
Missing 'Quantity' values imputed with median: 8.00
<class 'pandas.core.frame.DataFrame'>
Index: 31440 entries, 0 to 99998
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Transaction_ID      29860 non-null  object        
 1   Transaction_Date    31440 non-null  datetime64[ns]
 2   Customer_ID         29922 non-null  object        
 3   Product_Name        31440 non-null  object        
 4   Quantity            31440 non-null  float64       
 5   Price               31440 non-null  float64       
 6   Payment_Method      31440 non-null  object        
 7   Transaction_Status  26225 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(5)
memory usage: 2.2+ MB


### Feature Engineering

We will create new features from the `Transaction_Date` column to enrich the dataset for time-based analysis in Tableau. This includes extracting the year, month, day of the week, and hour of the transaction.

In [9]:
# Extract temporal features from 'Transaction_Date'
df['Transaction_Year'] = df['Transaction_Date'].dt.year
df['Transaction_Month'] = df['Transaction_Date'].dt.month
df['Transaction_Day'] = df['Transaction_Date'].dt.day
df['Transaction_DayOfWeek'] = df['Transaction_Date'].dt.dayofweek # Monday=0, Sunday=6
df['Transaction_Hour'] = df['Transaction_Date'].dt.hour

# Display the first few rows with new features
display(df.head())

# Display info to show new columns and their types
df.info()

,Transaction_ID,Transaction_Date,Customer_ID,Product_Name,Quantity,Price,Payment_Method,Transaction_Status,Transaction_Year,Transaction_Month,Transaction_Day,Transaction_DayOfWeek,Transaction_Hour
0,T0001,2024-08-02,C2205,Headphones,5.0,420.210000,paypal,NaN,2024,8,2,4,0
1,T0002,2020-02-10,C3156,Coffee,469.0,-445.342025,creditcard,pending,2020,2,10,0,0
3,T0004,2020-08-17,C3009,Tab,7.0,868.608341,paypal,pending,2020,8,17,0,0
5,T0006,2021-10-26,C4241,Smartphone,598.0,-54.398057,paypal,completed,2021,10,26,1,0
12,T0013,2020-03-18,C1692,Smartphone,564.0,-54.398057,cash,completed,2020,3,18,2,0


<class 'pandas.core.frame.DataFrame'>
Index: 31440 entries, 0 to 99998
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Transaction_ID         29860 non-null  object        
 1   Transaction_Date       31440 non-null  datetime64[ns]
 2   Customer_ID            29922 non-null  object        
 3   Product_Name           31440 non-null  object        
 4   Quantity               31440 non-null  float64       
 5   Price                  31440 non-null  float64       
 6   Payment_Method         31440 non-null  object        
 7   Transaction_Status     26225 non-null  object        
 8   Transaction_Year       31440 non-null  int32         
 9   Transaction_Month      31440 non-null  int32         
 10  Transaction_Day        31440 non-null  int32         
 11  Transaction_DayOfWeek  31440 non-null  int32         
 12  Transaction_Hour       31440 non-null  int32         
dtypes: dat

### Handle Missing `Transaction_ID` and `Customer_ID`

`Transaction_ID` and `Customer_ID` are unique identifiers. Missing values in these columns are problematic as imputation could lead to incorrect or duplicate IDs. For the purpose of this analysis, we will drop rows where these critical identifiers are missing. In a real-world scenario, you might investigate these further or try to reconstruct them if possible.

In [10]:
# Drop rows where 'Transaction_ID' or 'Customer_ID' are missing
initial_rows_before_dropping_ids = df.shape[0]
df.dropna(subset=['Transaction_ID', 'Customer_ID'], inplace=True)

print(f"Dropped {initial_rows_before_dropping_ids - df.shape[0]} rows with missing Transaction_ID or Customer_ID.")
print(f"New DataFrame shape: {df.shape}")

# Verify that there are no more missing Transaction_ID or Customer_ID
print('\nMissing values after handling IDs:')
print(df[['Transaction_ID', 'Customer_ID']].isnull().sum())

Dropped 3030 rows with missing Transaction_ID or Customer_ID.
New DataFrame shape: (28410, 13)

Missing values after handling IDs:
Transaction_ID    0
Customer_ID       0
dtype: int64


### Handle Missing `Transaction_Status`

The `Transaction_Status` column is categorical. Missing values here could represent an 'unknown' status or simply unrecorded data. Imputing with 'unknown' or the mode is a common strategy. Given that we want to prepare this for Tableau, explicit categories are better than NaNs. We will impute missing `Transaction_Status` values with 'unknown'.

In [12]:
# Impute missing 'Transaction_Status' with 'unknown'
df['Transaction_Status'] = df['Transaction_Status'].fillna('unknown')

# Display unique values to confirm 'unknown' is present
print('Unique values in Transaction_Status after imputation:')
print(df['Transaction_Status'].unique())

# Check for remaining missing values in the DataFrame
print('\nMissing values across all columns after handling:')
print(df.isnull().sum())

df.info()

Unique values in Transaction_Status after imputation:
['unknown' 'pending' 'completed' 'failed']

Missing values across all columns after handling:
Transaction_ID           0
Transaction_Date         0
Customer_ID              0
Product_Name             0
Quantity                 0
Price                    0
Payment_Method           0
Transaction_Status       0
Transaction_Year         0
Transaction_Month        0
Transaction_Day          0
Transaction_DayOfWeek    0
Transaction_Hour         0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
Index: 28410 entries, 0 to 99998
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Transaction_ID         28410 non-null  object        
 1   Transaction_Date       28410 non-null  datetime64[ns]
 2   Customer_ID            28410 non-null  object        
 3   Product_Name           28410 non-null  object        
 4   Quantity               28

### Handle Negative Prices

During the numerical data cleaning, we noted that the median price was negative, and negative prices can exist in the dataset. For most analytical purposes and for seamless integration with tools like Tableau, prices are typically expected to be positive. We will convert any negative `Price` values to their absolute value. This assumes that negative prices represent a magnitude of a transaction (e.g., a refund amount) rather than an actual negative monetary flow for a sale item. If the business logic dictates a different interpretation (e.g., treating them as specific refund entries), a more complex approach would be needed. For now, we will ensure all prices are non-negative.

In [13]:
# Convert any negative 'Price' values to their absolute value
initial_negative_prices = df[df['Price'] < 0].shape[0]
df['Price'] = df['Price'].abs()
print(f"Converted {initial_negative_prices} negative price values to positive.")

# Verify no more negative prices
print(f"Number of transactions with negative price after conversion: {df[df['Price'] < 0].shape[0]}")

# Display summary statistics for Price to confirm range
display(df['Price'].describe())

Converted 19042 negative price values to positive.
Number of transactions with negative price after conversion: 0


,Price
count,28410.000000
mean,366.222705
std,314.972135
min,50.130776
25%,54.398057
50%,283.994007
75%,644.006317
max,999.968903


### Final Data Review and Export

With all cleaning, preprocessing, and feature engineering steps completed, the DataFrame is now ready for use in Tableau. We will save the cleaned DataFrame to a new CSV file. This file can then be directly imported into Tableau for visualization and dashboard creation.

In [14]:
# Define the output file path
output_file_path = '/content/cleaned_financial_transactions.csv'

# Save the cleaned DataFrame to a CSV file
df.to_csv(output_file_path, index=False)

print(f"Cleaned data saved to {output_file_path}")
print("DataFrame is ready for Tableau!")

Cleaned data saved to /content/cleaned_financial_transactions.csv
DataFrame is ready for Tableau!


### Data Preprocessing and Cleaning for Tableau

To prepare this data for effective analysis and dashboard creation in Tableau, we will typically follow these steps:

1.  **Understand Data Types and Missing Values**: Examine the data types of each column and identify any missing values. This helps in deciding how to handle them (e.g., imputation, deletion).

2.  **Handle Duplicates**: Check for and remove any duplicate rows that might exist in the dataset.

3.  **Clean Text Data**: If there are text fields, clean them by removing extra spaces, standardizing casing, or correcting typos.

4.  **Date/Time Conversion**: Ensure all date and time columns are in the correct format. Tableau works well with `datetime` objects, so converting them in Python is beneficial.

5.  **Categorical Data Management**: Identify categorical variables. If they are represented numerically but are truly categories (e.g., `transaction_type_id`), consider mapping them to more descriptive labels. For Tableau, it's often best to have descriptive text rather than just IDs.

6.  **Numerical Data Validation**: Check numerical columns for outliers or incorrect values that might skew analysis.

7.  **Feature Engineering (Optional but Recommended)**: Create new features that might be useful for analysis. For financial transactions, this could include:
    *   `day_of_week`, `month`, `year` from transaction dates.
    *   `transaction_hour` to analyze hourly patterns.
    *   `transaction_age` (time since transaction).
    *   Categorical flags like `is_weekend`, `is_holiday`.

8.  **Data Aggregation (Optional)**: If the dataset is very large or if you need to analyze at a higher level (e.g., daily total sales), aggregate the data before exporting. However, for Tableau, it's often better to import granular data and perform aggregations within Tableau itself for flexibility.

After these steps, the cleaned DataFrame can be saved to a CSV file or other suitable format that Tableau can easily import.